In [1]:
import sys, json, logging, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
CONFIG = {
    'SEED': 42, 'TEST_SIZE': 0.2, 'CALIBRATION_SIZE': 0.1,
    'CATBOOST_ITERATIONS': 500, 'CATBOOST_LEARNING_RATE': 0.05, 'CATBOOST_DEPTH': 6,
    'MIN_POSITIVES': 30
}

In [3]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

In [4]:
# Load numeric-only features
X_list = []
for name in best_combo:
    df = pd.read_parquet(DATA_DIR / f"features_{name}.parquet").drop(columns=['organism'], errors='ignore')
    X_list.append(df)
X_full = pd.concat(X_list, axis=1)

# Save feature column order
with open(DATA_DIR / 'train_feature_columns.json', 'w') as f:
    json.dump(X_full.columns.tolist(), f)

groups = labeled_pd['organism']
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['SEED'])
train_idx, test_idx = next(gss.split(X_full, labeled_pd[active_stressors[0]], groups=groups))

# Save split
import json
with open(DATA_DIR / 'train_test_indices.json', 'w') as f:
    json.dump({'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist()}, f)

In [5]:
def train_stressor(stressor):
    """Train CatBoost classifier, filtering to organisms with experiment data.

    Organisms with all-zero labels for a stressor had no RB-TnSeq experiments
    under that condition — their zeros are missing data, not true negatives.
    Filtering to orgs_with_data removes ~100K false-negative proteins and raises
    effective positive rates from ~0.6% to ~1.8% for metals like Zn.
    """
    y_full = labeled_pd[stressor]

    # Restrict to organisms that actually ran experiments for this stressor.
    orgs_with_data = labeled_pd.groupby('organism')[stressor].sum()
    orgs_with_data = orgs_with_data[orgs_with_data > 0].index
    mask = labeled_pd['organism'].isin(orgs_with_data)

    y = y_full[mask]
    X = X_full[mask]
    g = groups[mask]

    if y.sum() < CONFIG['MIN_POSITIVES']:
        log.warning(f"{stressor}: only {int(y.sum())} positives after org-filter; skipping.")
        return None

    n_orgs = g.nunique()
    if n_orgs < 3:
        log.warning(f"{stressor}: only {n_orgs} organism(s) with data; too few to split reliably; skipping.")
        return None

    # Per-stressor split on the filtered subset (not the global train_idx/test_idx)
    gss_s = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'],
                              random_state=CONFIG['SEED'])
    s_train_idx, s_test_idx = next(gss_s.split(X, y, groups=g))

    X_tr, X_te = X.iloc[s_train_idx], X.iloc[s_test_idx]
    y_tr, y_te = y.iloc[s_train_idx], y.iloc[s_test_idx]
    g_tr = g.iloc[s_train_idx]

    if y_tr.nunique() < 2:
        log.warning(f"{stressor}: training set has only one class; skipping.")
        return None

    # Calibration split: only if training set has >= 2 organisms to group-split on
    platt = None
    if g_tr.nunique() >= 2:
        gss_cal = GroupShuffleSplit(n_splits=1, test_size=CONFIG['CALIBRATION_SIZE'],
                                    random_state=CONFIG['SEED'])
        sub_idx, cal_idx = next(gss_cal.split(X_tr, y_tr, groups=g_tr))
        X_sub, X_cal = X_tr.iloc[sub_idx], X_tr.iloc[cal_idx]
        y_sub, y_cal = y_tr.iloc[sub_idx], y_tr.iloc[cal_idx]
    else:
        log.warning(f"{stressor}: only 1 train organism; skipping calibration split.")
        X_sub, y_sub = X_tr, y_tr
        X_cal, y_cal = X_tr.iloc[:0], y_tr.iloc[:0]  # empty

    model = CatBoostClassifier(
        iterations=CONFIG['CATBOOST_ITERATIONS'],
        learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
        depth=CONFIG['CATBOOST_DEPTH'],
        cat_features=None, eval_metric='AUC',
        auto_class_weights='Balanced',
        random_seed=CONFIG['SEED'], verbose=50)
    model.fit(X_sub, y_sub, verbose=50)

    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    if len(y_cal) > 0 and y_cal.nunique() >= 2:
        cal_proba = model.predict_proba(X_cal)[:, 1]
        platt = LogisticRegression()
        platt.fit(cal_proba.reshape(-1, 1), y_cal)
        y_proba_cal = platt.predict_proba(y_proba.reshape(-1, 1))[:, 1]
    else:
        y_proba_cal = y_proba

    metrics = {
        'AUC': roc_auc_score(y_te, y_proba),
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_te, y_pred),
        'Pos rate': y_te.mean(),
        'n_orgs_train': int(g_tr.nunique()),
        'n_orgs_test': int(g.iloc[s_test_idx].nunique()),
    }

    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_final.cbm"))
    if platt is not None:
        with open(MODEL_DIR / f"stressor_{stressor}_platt.pkl", 'wb') as f:
            pickle.dump(platt, f)

    pd.DataFrame({
        'y_test': y_te.values, 'y_proba': y_proba,
        'y_proba_cal': y_proba_cal, 'group': g.iloc[s_test_idx].values,
    }).to_parquet(MODEL_DIR / f"stressor_{stressor}_predictions.parquet")

    return metrics

In [6]:
all_metrics = {}
for s in active_stressors:
    log.info(f"Training {s}")
    m = train_stressor(s)
    if m: all_metrics[s] = m

pd.DataFrame(all_metrics).T.to_csv(DATA_DIR / 'final_model_performance.csv')
log.info("Training complete.")

2026-06-29 02:43:31,748 INFO Training Zn


0:	total: 70.7ms	remaining: 35.3s


50:	total: 746ms	remaining: 6.57s


100:	total: 1.41s	remaining: 5.58s


150:	total: 2.08s	remaining: 4.8s


...

450:	total: 4.23s	remaining: 460ms


2026-06-29 02:47:23,442 INFO Training Biofilm


2026-06-29 02:47:23,452 WARNING Biofilm: only 1 organism(s) with data; too few to split reliably; skipping.


2026-06-29 02:47:23,457 INFO Training complete.


499:	total: 4.68s	remaining: 0us


### Regression training on continuous fitness scores (metals)

Requires `labeled_pd.parquet` to contain `{stressor}_fit` continuous columns,
produced when NB01 is re-run on Seaborg with the updated Cell 6 / Cell 12.

**Why regression for metals**: Binary labels collapse the full fitness distribution
to 0/1 at a -2.0 threshold, discarding the gradient between "strongly essential"
(fitness ≈ -5) and "slightly impaired" (fitness ≈ -1). Regression on the raw score
also sidesteps the class-imbalance problem: every tested protein contributes signal
regardless of whether it crosses the binary threshold.

**Evaluation**: Spearman ρ (rank correlation) is the primary metric — does the model
correctly rank metal resistance genes above housekeeping genes? AUC_from_ranking is
a secondary check: treating the top-ranked predictions as "positive" should recover
the known binary-positive genes.

In [ ]:
def train_stressor_regression(stressor):
    """Train CatBoost regressor on continuous RB-TnSeq fitness scores.

    Requires a '{stressor}_fit' column in labeled_pd (produced by NB01 re-run).
    Proteins with NaN fitness are excluded — they had no experiment for this stressor.
    """
    fit_col = f"{stressor}_fit"
    if fit_col not in labeled_pd.columns:
        log.warning(f"{stressor}: column '{fit_col}' absent — re-run NB01 on Seaborg first.")
        return None

    mask = labeled_pd[fit_col].notna()
    y = labeled_pd.loc[mask, fit_col].astype(float)
    X = X_full[mask]
    g = groups[mask]
    n_tested = int(mask.sum())

    if n_tested < CONFIG['MIN_POSITIVES'] * 10:
        log.warning(f"{stressor}: only {n_tested} proteins with data; skipping regression.")
        return None

    n_orgs = g.nunique()
    if n_orgs < 2:
        log.warning(f"{stressor}: only {n_orgs} organism(s) with data; need ≥2 to split; skipping.")
        return None

    log.info(f"{stressor}: {n_tested} proteins with fitness data across {n_orgs} organisms "
             f"({int((y < -2).sum())} binary positives)")

    gss_s = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'],
                              random_state=CONFIG['SEED'])
    s_train_idx, s_test_idx = next(gss_s.split(X, y, groups=g))

    X_tr, X_te = X.iloc[s_train_idx], X.iloc[s_test_idx]
    y_tr, y_te = y.iloc[s_train_idx], y.iloc[s_test_idx]

    model = CatBoostRegressor(
        iterations=CONFIG['CATBOOST_ITERATIONS'],
        learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
        depth=CONFIG['CATBOOST_DEPTH'],
        loss_function='RMSE',
        random_seed=CONFIG['SEED'], verbose=50)
    model.fit(X_tr, y_tr, verbose=50)

    y_pred = model.predict(X_te)
    rho, pval = spearmanr(y_te, y_pred)
    rmse = float(np.sqrt(np.mean((y_te.values - y_pred) ** 2)))

    # Secondary metric: does ranking by predicted fitness recover binary positives?
    y_binary_te = (y_te < -2).astype(int)
    auc_from_ranking = float(roc_auc_score(y_binary_te, -y_pred)) if y_binary_te.sum() > 0 else None

    metrics = {
        'Spearman_rho': float(rho),
        'Spearman_pval': float(pval),
        'RMSE': rmse,
        'AUC_from_ranking': auc_from_ranking,
        'n_tested': n_tested,
        'n_binary_pos': int((y < -2).sum()),
        'pos_rate': float((y < -2).mean()),
        'n_orgs': n_orgs,
    }

    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_regression.cbm"))
    pd.DataFrame({
        'y_test': y_te.values, 'y_pred': y_pred,
        'group': g.iloc[s_test_idx].values,
    }).to_parquet(MODEL_DIR / f"stressor_{stressor}_reg_predictions.parquet")

    return metrics


METAL_STRESSORS = ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'As', 'Hg', 'Pb', 'Mn', 'Fe', 'Se', 'Ag', 'Al']
reg_metrics = {}
for s in METAL_STRESSORS:
    log.info(f"Regression: {s}")
    m = train_stressor_regression(s)
    if m:
        reg_metrics[s] = m
        log.info(f"  Spearman rho={m['Spearman_rho']:.3f} (p={m['Spearman_pval']:.2e})  "
                 f"AUC_from_ranking={m['AUC_from_ranking']}")

if reg_metrics:
    pd.DataFrame(reg_metrics).T.to_csv(DATA_DIR / 'regression_model_metrics.csv')
    log.info(f"Regression metrics saved for {len(reg_metrics)} metal stressors.")
else:
    log.info("No regression results — re-run NB01 on Seaborg to generate '{stressor}_fit' columns.")

In [8]:
# Tune decision thresholds per stressor to maximize F1 score
best_thresholds = {}

for stressor in active_stressors:
    pred_file = MODEL_DIR / f"stressor_{stressor}_predictions.parquet"
    if not pred_file.exists():
        continue
    
    preds_df = pd.read_parquet(pred_file)
    y_test = preds_df['y_test'].values
    y_proba = preds_df['y_proba'].values
    
    # Sweep thresholds from 0.01 to 0.99
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_scores = []
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        f1_scores.append(f1)
    
    # Find threshold maximizing F1
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    best_thresholds[stressor] = float(best_threshold)
    log.info(f"{stressor}: best threshold={best_threshold:.3f}, F1={best_f1:.4f}")

# Save best thresholds
with open(DATA_DIR / 'best_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)

log.info(f"Saved best thresholds for {len(best_thresholds)} stressors to data/best_thresholds.json")

2026-06-29 02:48:24,961 INFO Zn: best threshold=0.380, F1=0.0747


2026-06-29 02:48:25,254 INFO Cu: best threshold=0.600, F1=0.1404


2026-06-29 02:48:25,746 INFO Cd: best threshold=0.010, F1=0.0172


2026-06-29 02:48:26,035 INFO Co: best threshold=0.500, F1=0.0635


2026-06-29 02:48:26,310 INFO Ni: best threshold=0.650, F1=0.0571


...

2026-06-29 02:48:36,952 INFO Nitrogen limitation: best threshold=0.010, F1=0.0000


2026-06-29 02:48:37,052 INFO Iron limitation: best threshold=0.210, F1=0.0366


2026-06-29 02:48:37,340 INFO UV: best threshold=0.740, F1=0.3532


2026-06-29 02:48:37,438 INFO Anaerobic: best threshold=0.030, F1=0.0882


2026-06-29 02:48:37,440 INFO Saved best thresholds for 46 stressors to data/best_thresholds.json
